In [1]:
import html
from pathlib import Path
from collections import defaultdict
import json 

full_path = 'data/card_creation_comparison/runs/confronto-02/reranking'

# Cerca il percorso anche nelle cartelle superiori alla directory di avvio.
cwd = Path.cwd().resolve()
ROOT = next(
    (base / full_path for base in (cwd, *cwd.parents)
     if (base / full_path).is_dir()),
    cwd / full_path,
)
paths = sorted(ROOT.glob('*.jsonl'))
if not paths:
    raise ValueError(f'No JSONL files found in {ROOT}')


In [ ]:
import runpy

top_n = 3
builder_path = next(
    base / 'Qualtrics/build_retrieval_survey.py'
    for base in ROOT.parents
    if (base / 'Qualtrics/build_retrieval_survey.py').is_file()
)
load_candidates = runpy.run_path(str(builder_path))['load_candidates']
settings = load_candidates(ROOT, top_n)
query_labels = {
    query_id: rows[0]['label']
    for _, queries in settings.values()
    for query_id, rows in queries.items()
}


In [ ]:
chunk_counts = defaultdict(int)

for configuration, (_, queries) in settings.items():
    for query_id, rows in queries.items():
        for row in rows:
            chunk_counts[(query_id, row['chunk_id'])] += 1

for (query_id, chunk_id), count in sorted(chunk_counts.items()):
    print(f'Label: {query_labels[query_id]}, Query: {query_id}, Chunk: {chunk_id}, Configurations: {count}')


In [ ]:
print("\nChunks selected in at least 2 configurations:")

shared_count = 0
single_count = 0
for (query_id, chunk_id), count in sorted(chunk_counts.items()):
    if count >= 2:
        shared_count += 1
        print(f'Label: {query_labels[query_id]}, Query: {query_id}, Chunk: {chunk_id}, Configurations: {count}')
    else:
        single_count += 1
print(f'Total unique query-chunk pairs: {len(chunk_counts)}')
print(f'Total pairs selected in at least 2 configurations: {shared_count}')
print(f'Total pairs selected in only 1 configuration: {single_count}')


In [ ]:
for configuration, (_, queries) in settings.items():
    print(f'\nConfiguration: {configuration}')
    for query_id, rows in queries.items():
        print(f'  Query: {query_id}, Label: {rows[0]["label"]}')
        for row in rows:
            print(f'    Chunk ID: {row["chunk_id"]}, Rank: {row["reranker_rank"]}')
